# 02 - XGBoost Modeling

This notebook trains an XGBoost model for sales prediction using the shared
feature pipeline from `src.data.features`. It performs:
1. Data loading and deduplication
2. Feature engineering via the shared pipeline
3. Train/Test temporal split
4. XGBoost training and evaluation
5. TimeSeriesSplit cross-validation
6. Artifact saving

In [ ]:
import sys
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import json
import joblib
import datetime

# Ensure project root is on sys.path for imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.features import build_feature_pipeline

# Paths
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "sample_features.csv"
FORECAST_PATH = PROJECT_ROOT / "models" / "artifacts" / "xgb_forecast_final.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics" / "xgb_metrics_final.json"
MODEL_PATH = PROJECT_ROOT / "models" / "artifacts"

FORECAST_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Load & clean data ---
df = pd.read_csv(FEATURES_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

# Handle duplicate dates by averaging
if df['date'].duplicated().sum() > 0:
    n_dup = df['date'].duplicated().sum()
    print(f"[INFO] {n_dup} doublons detectes, moyenne par date")
    df = df.groupby('date').mean(numeric_only=True).reset_index()

# --- Feature engineering using the SHARED pipeline ---
# This ensures feature names match those used by train.py and predict.py
lags = [1, 7, 14]
windows = [7, 14]

df = build_feature_pipeline(df, lags=lags, windows=windows)

# Drop NaN rows introduced by lags/rolling
df_clean = df.dropna().reset_index(drop=True)

y = df_clean['value']
X = df_clean.drop(columns=['value', 'date'])

print(f"[INFO] Shape apres creation features : X={X.shape}, y={y.shape}")
print(f"[INFO] Feature columns: {list(X.columns)}")

# --- Train/Test split (temporal) ---
horizon = 30

X_train, X_test = X.iloc[:-horizon], X.iloc[-horizon:]
y_train, y_test = y.iloc[:-horizon], y.iloc[-horizon:]
dates_test = df_clean['date'].iloc[-horizon:]

print(f"[INFO] Train: {len(y_train)} samples, Test: {len(y_test)} samples")

# --- Train XGBoost ---
model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# --- Predict ---
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# --- Metrics train/test ---
metrics = {
    "MAE_train": float(mean_absolute_error(y_train, y_pred_train)),
    "RMSE_train": float(np.sqrt(mean_squared_error(y_train, y_pred_train))),
    "MAE_test": float(mean_absolute_error(y_test, y_pred_test)),
    "RMSE_test": float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
    "num_train_samples": len(y_train),
    "num_test_samples": len(y_test)
}

print("Metrics train/test XGBoost :", metrics)

# --- Cross-validation TimeSeriesSplit ---
tscv = TimeSeriesSplit(n_splits=5)
cv_mae = []

for train_idx, test_idx in tscv.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[test_idx]
    model_cv = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
    model_cv.fit(X_tr, y_tr)
    y_pred_cv = model_cv.predict(X_val)
    mae_fold = float(mean_absolute_error(y_val, y_pred_cv))
    cv_mae.append(mae_fold)

metrics['CV_mae_mean'] = float(np.mean(cv_mae))
metrics['CV_mae_folds'] = cv_mae
print("\n Metrics CV TimeSeriesSplit :", metrics['CV_mae_mean'])

# --- Save forecast ---
forecast_train = df_clean.iloc[:-horizon][['date']].copy()
forecast_train['yhat'] = y_pred_train
forecast_train['value'] = y_train.values

forecast_test = df_clean.iloc[-horizon:][['date']].copy()
forecast_test['yhat'] = y_pred_test
forecast_test['value'] = y_test.values

forecast = pd.concat([forecast_train, forecast_test], axis=0).reset_index(drop=True)

forecast.to_csv(FORECAST_PATH, index=False)
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

# --- Save model artifact (consistent with train.py naming) ---
today = datetime.datetime.now().strftime('%Y%m%d')
model_file = MODEL_PATH / f"model_{today}_baseline.pkl"
joblib.dump(model, model_file)

print(f"Model sauvegarde : {model_file}")
print(f"Forecast sauvegarde : {FORECAST_PATH}")
print(f"Metrics sauvegardees : {METRICS_PATH}")
